<img src="logo.png" alt="Vegeta" width="240">

# Mellonia — manufacturability with PrusaSlicer

Mellonia is standalone: any STL, explicit settings, an orientation chosen by you.
The STL here is written with a few lines of Python so no CAD package is needed.

In [ ]:
from pathlib import Path
import itertools
from vegeta import mellonia
from vegeta.mellonia import Orientation, PrintSettings, slice_stl

RUNS = Path("_runs/mellonia"); RUNS.mkdir(parents=True, exist_ok=True)

def box_stl(path, dx, dy, dz):
    v = {c: (c[0] * dx, c[1] * dy, c[2] * dz) for c in itertools.product((0, 1), repeat=3)}
    quads = [((0,0,0),(0,1,0),(1,1,0),(1,0,0)), ((0,0,1),(1,0,1),(1,1,1),(0,1,1)), ((0,0,0),(1,0,0),(1,0,1),(0,0,1)),
             ((0,1,0),(0,1,1),(1,1,1),(1,1,0)), ((0,0,0),(0,0,1),(0,1,1),(0,1,0)), ((1,0,0),(1,1,0),(1,1,1),(1,0,1))]
    out = ["solid box"]
    for a, b, c, d in quads:
        for tri in ((a, b, c), (a, c, d)):
            out += ["facet normal 0 0 0", "outer loop"] + [f"vertex {x} {y} {z}" for x, y, z in (v[p] for p in tri)] + ["endloop", "endfacet"]
    Path(path).write_text("\n".join(out + ["endsolid box"]) + "\n")
    return Path(path)

part = box_stl(RUNS / "post.stl", 10, 10, 40)

## 1. Explicit settings (PrusaSlicer option names)

In [ ]:
settings = PrintSettings(
    name="example PLA 0.2",
    printer={"bed_shape": "0x0,250x0,250x210,0x210", "max_print_height": 210, "nozzle_diameter": 0.4},
    filament={"filament_diameter": 1.75, "filament_density": 1.24, "filament_cost": 25,
              "temperature": 210, "bed_temperature": 60},
    print={"layer_height": 0.2, "first_layer_height": 0.2, "perimeters": 2, "fill_density": "20%"},
)

## 2. Slice in the orientation the engineer chooses

In [ ]:
upright = slice_stl(part, settings, Orientation(), RUNS / "upright")
upright

## 3. Try another orientation — explicitly, no optimiser

In [ ]:
lying = slice_stl(part, settings, Orientation(rotate_x=90), RUNS / "lying")
for name, r in (("upright", upright), ("lying", lying)):
    if not r.ok:
        print(name, "FAILED:", r.messages)
        continue
    m = r.metrics
    print(f"{name:8s} layers={m['layer_count']:4d}  time={m['estimated_time']:>10s}  filament={m['filament_used_g']} g")

In [ ]:
fig = mellonia.plot_layers(lying) if lying.ok else None

### The toolpath
The sliced part as PrusaSlicer will print it: 3D toolpath coloured by layer, and single layers from above (perimeters, infill, travel moves).

In [ ]:
from vegeta.mellonia import viz

if lying.ok:
    viz.show(viz.plot_toolpath(lying))

In [ ]:
if lying.ok:
    fig = viz.plot_layer(lying, 10)
    fig = viz.plot_layer_grid(upright, n=6)

## 4. Mistakes are reported, not ignored

In [ ]:
typo = slice_stl(part, settings.replace(print={"layer_heigth": 0.3}), Orientation(), RUNS / "typo")
typo.status, typo.messages[0]

Everything PrusaSlicer actually used is kept next to the G-code:

In [ ]:
print(Path(upright.artifacts['effective_config']).read_text()[:400])